# Импорты

In [1]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
from pathlib import Path
import pytorch_lightning as pl
from omegaconf import OmegaConf
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

from src.training.lightning_module import GPTLightningModule
from src.data.wikitext_datamodule import WikiTextDataModule
from src.tokenization.tokenizers import load_bpe_tokenizer

import warnings
warnings.filterwarnings("ignore", message=".*LeafSpec.*")

# Если планируете смотреть TensorBoard прямо в ноутбуке
# %load_ext tensorboard

W0827 20:45:00.814000 29764 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# ClearML

In [2]:
from clearml import Task
from dotenv import load_dotenv
load_dotenv()

task = Task.init(project_name="GPT-training", task_name="Training_13", continue_last_task=True)

ClearML Task: continuing previous task id=3d3ba2eb2717425a82ded6be9d56db61 Notice this run will not be reproducible!
ClearML results page: https://app.clear.ml/projects/387256ae78834925933b20412046ed97/tasks/3d3ba2eb2717425a82ded6be9d56db61/output/log


# конфиг

In [4]:
config = OmegaConf.load("../configs/model_config.yaml")

# print(OmegaConf.to_yaml(config))  # удобно сразу видеть, с чем запускаемся
task.connect(OmegaConf.to_container(config, resolve=True))

{'model': {'vocab_size': 10000,
  'd_model': 512,
  'n_heads': 8,
  'n_layers': 6,
  'd_ff': 2048,
  'max_len': 512},
 'training': {'batch_size': 16,
  'learning_rate': 0.0003,
  'weight_decay': 0.1,
  'warmup_steps': 1000,
  'max_epochs': 3,
  'gradient_clip_val': 1.0,
  'optimizer': {'name': 'adamw', 'betas': [0.9, 0.999], 'eps': 1e-08},
  'scheduler': {'name': 'cosine', 'T_max': 52044, 'eta_min': 1e-06}},
 'data': {'max_length': 512, 'batch_size': 16, 'num_workers': 0},
 'paths': {'data_dir': 'R:/Folders/1. master_degree_ITMO/semester 2/LABS/MNNA-2026/MNNA-2026-labs-DmitrievDM/data/processed/wikitext/stage2_filtered',
  'tokenizer_path': 'R:/Folders/1. master_degree_ITMO/semester 2/LABS/MNNA-2026/MNNA-2026-labs-DmitrievDM/data/tokenizers/bpe_tokenizer.json',
  'checkpoint_dir': 'R:/Folders/1. master_degree_ITMO/semester 2/LABS/MNNA-2026/MNNA-2026-labs-DmitrievDM/checkpoints'}}

# Модель и данные

In [5]:
model = GPTLightningModule(config)
datamodule = WikiTextDataModule(config)

# Полезная проверка перед долгим обучением — сколько параметров в модели
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

Total parameters: 29,164,304


# Callbacks и logger

In [6]:
checkpoint_callback = ModelCheckpoint(
    dirpath="../checkpoints/",
    filename="gpt-{epoch:02d}-{val_loss:.3f}",
    monitor="val_loss",
    mode="min",
    save_top_k=3,
    save_last=True,
)

lr_monitor = LearningRateMonitor(logging_interval="step")
tb_logger = TensorBoardLogger(save_dir="../logs/", name="gpt_notebook")

# sanity check

In [6]:
# sanity_trainer = pl.Trainer(
#     fast_dev_run=True,   # прогоняет 1 train + 1 val батч и останавливается
#     accelerator="gpu",
#     devices=1,
# )
# sanity_trainer.fit(model, datamodule=datamodule)
# print("Sanity check passed ✓")

# запуск TensorBoard

In [7]:
%load_ext tensorboard
# %reload_ext tensorboard
%tensorboard --logdir ../logs/

Reusing TensorBoard on port 6007 (pid 26040), started 3 days, 21:53:25 ago. (Use '!kill 26040' to kill it.)

# Обучение

In [8]:
trainer = pl.Trainer(
    max_epochs=config.training.max_epochs,
    accelerator="gpu",
    devices=1,  # или больше, если есть несколько GPU
    precision="16-mixed", # Было bf16-mixed
    gradient_clip_val=1.0,
    gradient_clip_algorithm="norm",
    callbacks=[checkpoint_callback, lr_monitor],
    logger=tb_logger,
    log_every_n_steps=10,
)

trainer.fit(model, datamodule=datamodule)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
r:\Folders\1. master_degree_ITMO\semester 2\LABS\MNNA-2026\MNNA-2026-labs-DmitrievDM\.venv\Lib\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ GPTModel │ 29.2 M │ train │     0 │
└───┴───────┴──────────┴────────┴───────┴───────┘

Trainable params: 29.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 29.2 M                                                                                               
Total estimated model params size (MB): 116.657                                                                    
Modules in train mode: 84                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

r:\Folders\1. master_degree_ITMO\semester 
2\LABS\MNNA-2026\MNNA-2026-labs-DmitrievDM\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connec
tor.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

r:\Folders\1. master_degree_ITMO\semester 
2\LABS\MNNA-2026\MNNA-2026-labs-DmitrievDM\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connec
tor.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

ClearML Monitor: Reporting detected, reverting back to iteration based reporting


2026-08-26 00:15:49,359 - clearml - INFO - inf value encountered. Reporting it as '0.0'. Use clearml.Logger.set_reporting_inf_value to assign another value


`Trainer.fit` stopped: `max_epochs=3` reached.


# Результаты

In [8]:
print(f"Best checkpoint: {checkpoint_callback.best_model_path}")
print(f"Best val_loss: {checkpoint_callback.best_model_score}")

import math
print(f"Best val_perplexity: {math.exp(checkpoint_callback.best_model_score):.2f}")

Best checkpoint: 
Best val_loss: None


TypeError: must be real number, not NoneType

# демонстрация restore + генерации

In [9]:
import omegaconf

loaded_model = GPTLightningModule.load_from_checkpoint(checkpoint_callback.best_model_path, weights_only=False)
loaded_model.eval()
loaded_model = loaded_model.to("cuda")
tokenizer_path = config.paths.tokenizer_path

generated_text = loaded_model.generate(
    prompt="The history",
    tokenizer=load_bpe_tokenizer(tokenizer_path),
    max_length=150,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
)
print(generated_text)

PermissionError: [Errno 13] Permission denied: 'r:/Folders/1. master_degree_ITMO/semester 2/LABS/MNNA-2026/MNNA-2026-labs-DmitrievDM/notebooks'

# демонстрация restore + продолжения обучения

In [10]:
resume_trainer = pl.Trainer(
    max_epochs=config.training.max_epochs + 1,
    accelerator="gpu", devices=1,
    precision="16-mixed",
    gradient_clip_val=1.0,
    callbacks=[checkpoint_callback, lr_monitor],
    logger=tb_logger,
)
resume_trainer.fit(model, datamodule=datamodule, ckpt_path=checkpoint_callback.best_model_path)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


ValueError: `.fit()` found no path for the best weights: ''. Please specify a path for a checkpoint `.fit(ckpt_path=PATH)`

In [ ]:
task.close()

In [ ]:
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()